# Figure 3

Paper panels with figure-specific PDF and CSV exports. Run from top to bottom. Original analysis notebooks are preserved.

## Setup

In [3]:
from pathlib import Path
from tempfile import TemporaryDirectory
from contextlib import contextmanager
from IPython.display import IFrame, display
from matplotlib.transforms import Bbox
import itertools
import os
import shutil
import pandas as pd
import KinematicPlot as kp
from group_config_new import build_groups
from survival_stats_runner import SurvivalStatsRunner

# All persistent exports belong to this paper figure, with one folder per panel.
ROOT = Path.cwd()
if not (ROOT / "KinematicPlot.py").is_file():
    raise RuntimeError("Run this notebook from the repository root.")
FIGURE_NUMBER = 3
NOTEBOOK_OUTPUT_DIR = ROOT / "Figures" / f"Figure{FIGURE_NUMBER}"
SC_DATA_DIR = ROOT / "SC data"
N_PERM = 20000
plotter = kp.PlotCreator()
stats_runner = SurvivalStatsRunner(tau=0.71, random_state=0, platform_offset=0.03, radius=0.07, fps=250)

def output_folder(*parts):
    # Create output folders only when the notebook is executed.
    folder = NOTEBOOK_OUTPUT_DIR.joinpath(*parts)
    folder.mkdir(parents=True, exist_ok=True)
    return folder

def panel_prefix(panel, name):
    return output_folder(f"Figure{panel}") / f"Figure{panel}_{name}"

def show_pdf(path, width=950, height=700):
    display(IFrame(src=Path(path).relative_to(ROOT).as_posix(), width=width, height=height))

@contextmanager
def working_directory(folder):
    # Preserve the working directory for legacy notebook loader calls.
    previous = Path.cwd()
    os.chdir(folder)
    try:
        yield
    finally:
        os.chdir(previous)

In [4]:
# Shared WT tracking QC thresholds for angle and TT trajectory analyses.
# Change WT_QC_ERROR_MAX here to adjust the reprojection-error cutoff for both.
WT_QC_ERROR_MAX = 30
WT_QC_SCORE_MIN = 0.8
WT_QC_MIN_CAMERAS = 2
WT_QC_MAX_INTERP_GAP_S = 0.02
WT_QC_MAX_INVALID_FRACTION = 0.3
WT_QC_MIN_VALID_FRACTION = 1.0 - WT_QC_MAX_INVALID_FRACTION

# Smooth after QC/interpolation with the utilities exponential moving average.
WT_ANGLE_SMOOTH = True
WT_ANGLE_SMOOTH_ALPHA = 0.4

WT_ANGLE_QC_KWARGS = dict(
    apply_tracking_qc=True,
    min_cameras=WT_QC_MIN_CAMERAS,
    max_interp_gap_s=WT_QC_MAX_INTERP_GAP_S,
    min_valid_fraction=WT_QC_MIN_VALID_FRACTION,
    error_max=WT_QC_ERROR_MAX,
    score_min=WT_QC_SCORE_MIN,
    smooth_angle=WT_ANGLE_SMOOTH,
    smooth_alpha=WT_ANGLE_SMOOTH_ALPHA,
)

WT_TT_QC_KWARGS = dict(
    apply_tracking_qc=True,
    min_cameras=WT_QC_MIN_CAMERAS,
    max_interp_gap_s=WT_QC_MAX_INTERP_GAP_S,
    min_valid_fraction=WT_QC_MIN_VALID_FRACTION,
    error_max=WT_QC_ERROR_MAX,
    score_min=WT_QC_SCORE_MIN,
)

WT_ANGLE_QC_KWARGS, WT_TT_QC_KWARGS

({'apply_tracking_qc': True,
  'min_cameras': 2,
  'max_interp_gap_s': 0.02,
  'min_valid_fraction': 0.7,
  'error_max': 30,
  'score_min': 0.8,
  'smooth_angle': True,
  'smooth_alpha': 0.4},
 {'apply_tracking_qc': True,
  'min_cameras': 2,
  'max_interp_gap_s': 0.02,
  'min_valid_fraction': 0.7,
  'error_max': 30,
  'score_min': 0.8})

In [5]:
def export_generated(prefix, default_panel, name, routes=None, pdfs=None):
    # Copy every generated CSV; route selected files to their actual figure panel.
    # Explicit PDF selection excludes extra figures produced by legacy functions.
    routes = routes or {}
    exported = []
    for source in sorted(prefix.parent.glob(prefix.name + "*")):
        suffix = source.name[len(prefix.name):]
        if source.suffix == ".pdf" and pdfs is not None and suffix not in pdfs:
            continue
        if source.suffix not in (".csv", ".pdf"):
            continue
        panel = routes.get(suffix, default_panel)
        target = Path(str(panel_prefix(panel, name)) + suffix)
        shutil.copy2(source, target)
        exported.append(target)
    return exported

def export_axes(fig, axes, destination):
    # Export existing plotted artists without recalculating data or statistics.
    # Hide other axes so adjacent panels cannot appear in the exported PDF.
    selected = list(axes)
    visibility = {ax: ax.get_visible() for ax in fig.axes}
    text_visibility = {label: label.get_visible() for label in fig.texts}
    try:
        for ax in fig.axes:
            ax.set_visible(ax in selected)
        # Combined-figure titles do not describe an individually exported panel.
        for label in fig.texts:
            label.set_visible(False)
        fig.canvas.draw()
        renderer = fig.canvas.get_renderer()
        bounds = Bbox.union([ax.get_tightbbox(renderer) for ax in selected])
        bounds = bounds.transformed(fig.dpi_scale_trans.inverted()).padded(0.08)
        fig.savefig(destination, bbox_inches=bounds, dpi=300)
    finally:
        for ax, visible in visibility.items():
            ax.set_visible(visible)
        for label, visible in text_visibility.items():
            label.set_visible(visible)

## WT T1/T2 groups and behavioral annotations

In [7]:
groups = build_groups(group_keys=["WT_T1_TTa", "WT_T2_TTa"], skip_missing=False, require_kinematics=False)
wt_tita_sc_paths = {"T2": SC_DATA_DIR / "WT-T2-TiTa_LegContact.csv"}
it_filtered_ll_path = r"C:\Users\wayne\OneDrive\Desktop\AnalysisAndFigures\Metadata\WT-T2-TiTa_new_IT_filtered.xlsx"
ot_filtered_ll_path = r"C:\Users\wayne\OneDrive\Desktop\AnalysisAndFigures\Metadata\WT-T2-TiTa_new_OT_filtered.xlsx"
wt_t2_tita_sc_path = SC_DATA_DIR / "WT-T2-TiTa_LegContact.csv"

# This preserves the filtered-file convention used in the main script.
wt_t2_tita_it_ot_sources = {
    "IT": {"path": ot_filtered_ll_path, "selection_mode": "numeric"},
    "OT": {"path": it_filtered_ll_path, "selection_mode": "numeric"},
}

wt_t2_tita_it_ot_sources
bo_filtered_ll_path = r"C:\Users\wayne\OneDrive\Desktop\AnalysisAndFigures\Metadata\WT-T1-TiTa_new_BO_filtered.xlsx"
nb_filtered_ll_path = r"C:\Users\wayne\OneDrive\Desktop\AnalysisAndFigures\Metadata\WT-T1-TiTa_new_NB_filtered.xlsx"
wt_t1_tita_sc_path = SC_DATA_DIR / "WT-T1-TiTa_LegContact.csv"

wt_t1_tita_bo_nb_sources = {
    "BO": {"path": nb_filtered_ll_path, "selection_mode": "numeric"},
    "NB": {"path": bo_filtered_ll_path, "selection_mode": "numeric"},
}

wt_t1_tita_bo_nb_sources

{'BO': {'path': 'C:\\Users\\wayne\\OneDrive\\Desktop\\AnalysisAndFigures\\Metadata\\WT-T1-TiTa_new_NB_filtered.xlsx',
  'selection_mode': 'numeric'},
 'NB': {'path': 'C:\\Users\\wayne\\OneDrive\\Desktop\\AnalysisAndFigures\\Metadata\\WT-T1-TiTa_new_BO_filtered.xlsx',
  'selection_mode': 'numeric'}}

## Figures 3B, 3C, 3D - IT_OT: angles, LP, KM

In [9]:
# Calculate each behavioral comparison once and retain every generated CSV.
with TemporaryDirectory(prefix="figure3_behavior_") as temporary:
    prefix = Path(temporary) / "behavior"
    plotter.plot_it_ot_landing_probability_and_latency(
        group_info=groups["WT_T2_TTa"],
        behavior_sources=wt_t2_tita_it_ot_sources,
        file_name=str(prefix),
        behavior_labels=("IT", "OT"),
        behavior_display_names={"IT": "Inward touch", "OT": "Outward touch"},
        trial_types=("Landing", "Flying"),
        tau=0.71,
        n_perm=N_PERM,
        contacted_leg="R-m",
        angle_start_s=-0.1,
        angle_end_s=0.1,
        target_fps=250,
        colors={"IT": "#8FD694", "OT": "#C7A0E8"},
        **WT_ANGLE_QC_KWARGS,
    )
    export_generated(prefix, '3C', 'IT_OT', routes={'_FT_angle_trace.pdf': '3B', '_FT_angle_traces.csv': '3B', '_FT_angle_qc_summary.csv': '3B', '_FT_angle_qc_skipped_trials.csv': '3B', '_landing_latency_inverted_KM.pdf': '3D', '_km_stats.csv': '3D', '_km_logrank.csv': '3D'})
# Trial selection is shared by LP and KM; retain the contributing rows with both.
shutil.copy2(str(panel_prefix('3C', 'IT_OT')) + "_trial_data.csv",
             str(panel_prefix('3D', 'IT_OT')) + "_trial_data.csv")
show_pdf(str(panel_prefix('3B', 'IT_OT')) + "_FT_angle_trace.pdf")
show_pdf(str(panel_prefix('3C', 'IT_OT')) + "_landing_probability.pdf")
show_pdf(str(panel_prefix('3D', 'IT_OT')) + "_landing_latency_inverted_KM.pdf")

## Figures 3F, 3G, 3H - BO_NB: angles, LP, KM

In [11]:
# Calculate each behavioral comparison once and retain every generated CSV.
with TemporaryDirectory(prefix="figure3_behavior_") as temporary:
    prefix = Path(temporary) / "behavior"
    plotter.plot_it_ot_landing_probability_and_latency(
        group_info=groups["WT_T1_TTa"],
        behavior_sources=wt_t1_tita_bo_nb_sources,
        file_name=str(prefix),
        behavior_labels=("BO", "NB"),
        behavior_display_names={"BO": "BO", "NB": "NB"},
        trial_types=("Landing", "Flying"),
        tau=0.71,
        n_perm=N_PERM,
        contacted_leg="R-f",
        angle_start_s=-0.1,
        angle_end_s=0.1,
        target_fps=250,
        colors={"BO": "#66C2A5", "NB": "#B39DDB"},
        **WT_ANGLE_QC_KWARGS,
    )
    export_generated(prefix, '3G', 'BO_NB', routes={'_FT_angle_trace.pdf': '3F', '_FT_angle_traces.csv': '3F', '_FT_angle_qc_summary.csv': '3F', '_FT_angle_qc_skipped_trials.csv': '3F', '_landing_latency_inverted_KM.pdf': '3H', '_km_stats.csv': '3H', '_km_logrank.csv': '3H'})
# Trial selection is shared by LP and KM; retain the contributing rows with both.
shutil.copy2(str(panel_prefix('3G', 'BO_NB')) + "_trial_data.csv",
             str(panel_prefix('3H', 'BO_NB')) + "_trial_data.csv")
show_pdf(str(panel_prefix('3F', 'BO_NB')) + "_FT_angle_trace.pdf")
show_pdf(str(panel_prefix('3G', 'BO_NB')) + "_landing_probability.pdf")
show_pdf(str(panel_prefix('3H', 'BO_NB')) + "_landing_latency_inverted_KM.pdf")

## Figures 3J and 3K - T2 path efficiency by outcome and versus latency

In [13]:
# One T2 calculation supplies both panels and their existing inferential tests.
with TemporaryDirectory(prefix="figure3_metrics_") as temporary:
    prefix = Path(temporary) / "path_efficiency"
    fig, axes, metric_df, stat_df = plotter.plot_TT_summary_metrics_vs_LL(
        group_info=groups["WT_T2_TTa"], tau=0.71, file_name=str(prefix),
        sc_csv_path=wt_tita_sc_paths["T2"], n_perm=N_PERM, **WT_TT_QC_KWARGS,
    )
    if fig is None:
        raise ValueError("No QC-passing T2 path-efficiency data are available.")
    export_axes(fig, [axes[1]], panel_prefix("3J", "path_efficiency_success_failed").with_suffix(".pdf"))
    export_axes(fig, [axes[0]], panel_prefix("3K", "path_efficiency_vs_LL").with_suffix(".pdf"))
    export_generated(prefix, "3J", "path_efficiency", routes={"_trend_stats.csv": "3K"}, pdfs=set())
    metric_df.to_csv(str(panel_prefix("3K", "path_efficiency")) + "_data.csv", index=False)
show_pdf(panel_prefix("3J", "path_efficiency_success_failed").with_suffix(".pdf"))
show_pdf(panel_prefix("3K", "path_efficiency_vs_LL").with_suffix(".pdf"))